In [ ]:
import cv2
import numpy as np
import os

# ============================================
# CONFIGURAÇÕES (ALTERE AQUI FACILMENTE)
# ============================================

# Nomes das pessoas (conforme seus arquivos)
pessoas = ['caio', 'edson', 'nicolas']

# Variações: com e sem avatar
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens
extensao = '.jpg'

# Parâmetros do Highboost
gaussian_kernel = 5          # Tamanho do kernel do filtro gaussiano (ímpar)
sigma = 0                     # Desvio padrão (0 = automático)
k_values = [1, 3]             # Fatores de realce (dois níveis)

# ============================================
# PROCESSAMENTO
# ============================================

print("Iniciando filtragem Highboost em imagens coloridas...")

for pessoa in pessoas:
    for versao in versoes:
        # Nome do arquivo de entrada (ex: caio_sem_avatar.jpg)
        nome_entrada = f"{pessoa}_{versao}{extensao}"

        if not os.path.exists(nome_entrada):
            print(f"Arquivo não encontrado: {nome_entrada}. Pulando...")
            continue

        # Carregar imagem colorida
        img_color = cv2.imread(nome_entrada)
        if img_color is None:
            print(f"Erro ao ler {nome_entrada}")
            continue

        # Converter para YUV (Y = luminância, U e V = crominância)
        img_yuv = cv2.cvtColor(img_color, cv2.COLOR_BGR2YUV)

        # Separar os canais
        y, u, v = cv2.split(img_yuv)

        # Converter Y para float32 para precisão
        y_float = np.float32(y)

        # Aplicar suavização gaussiana no canal Y
        blurred = cv2.GaussianBlur(y_float, (gaussian_kernel, gaussian_kernel), sigma)

        # Calcular a máscara (detalhes = Y original - borrado)
        mask = y_float - blurred

        # Aplicar para cada valor de k
        for k in k_values:
            # Highboost no canal Y: Y' = Y + k * mask
            y_sharp = y_float + k * mask

            # Recortar para o intervalo [0,255] e converter de volta para uint8
            y_sharp = np.clip(y_sharp, 0, 255).astype(np.uint8)

            # Recombinar os canais
            img_sharp_yuv = cv2.merge([y_sharp, u, v])

            # Converter de volta para BGR (colorido)
            img_sharp = cv2.cvtColor(img_sharp_yuv, cv2.COLOR_YUV2BGR)

            # Salvar a imagem resultante
            nome_saida = f"highboost_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, img_sharp)
            print(f"Salvo: {nome_saida}")

print("Processamento concluído!")